# AIFarmer CRISP-DM

This python notebook documents and runs the process of data understanding, gathering, and preparation for the Mlimi digital project AIFarmer. This acts as both a document for experimentation as well as documentation and a model for ACADES to continue development of AIFarmer or future Mlimi digital projects. Data for this project is acquired from a number of different sources in different formats for multiple different purposes:
1. Fine-tuning/training data for the creation of a specialized chatbot with competency in both English and Chichewa, as well as specialized expertise in Malawian agriculture.
    - Specifically, experience data for a reinforcement learning approach to fine-tuning (e.g. Deepseek D1 paper)
2. Image datasets of crops grown in Malawi, both healthy and diseased/pest-ridden, for training or fine-tuning CNNs or other vision-based AI models.

In [1]:
# Import statements
import pandas as pd
import numpy as np
import re
import json
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier 
from nltk import word_tokenize, ngrams

In [2]:
# Parameter variables to simplify using the run all command. Set one or both to true/false to run the code preparing each respective data type.
experience = True
vision = False

The experience data in this case is to be stored in a csv of the form:
query_id | query | query_lang | model_id | response_text | think_text | answer_text | score | scorer

## DigitalGreen Agriqueries
https://huggingface.co/datasets/DigiGreen/Kenya_Agri_queries

Description:
The open source "Kenya_Agri_Queries" dataset from huggingface contains a large sample of question/query-answer pairs from actual user data of a similar app to AIFarmer over a 10 month period, from September 2023 until June 2024. It contains good user data in the sense that these are actual questions that smallholder farms have asked, however it is limited in that the test ran in Kenya and all pairs are entirely in english. Relevant sticking points for data cleaning to look out for are:
- Region-specific queries (Cleaning for mentions of Kenya or regions in Kenya may be smart?)
- Quality control: Some questions and/or answers aren't useful (Remove "this question is out of my scope" and manual cleaning?)
- Non-Malawian crops (e.g. coffee is not grown here, remove mentions of coffee)

In [3]:
# Load the data with pandas and print head

In [4]:
# Print size, format, and type

## Opportunity International/ dmatekenya Chichewa Machine Translation
https://huggingface.co/datasets/dmatekenya/chichewa-machine-translation

Description:
The open source "chichewa-machine-translation" dataset from huggingface contains 14,002 rows of machine translated Chichewa questions from a variety of topics and their corresponding English translations. While not specified, it's likely that the questions were written in English and were machine translated to Chichewa. In addition to the en/ch question pairs, each row has an id, a topic, and a source. Given the huggingface profile of dmatekenya, it's likely that this was related to their work gathering data and training models with Opportunity International.
TODO: Follow-up after meeting with Opportunity International

Outputs: 
- processed_data.csv is a collected csv of all english and chichewa samples split into separate rows with corresponding sent_id to their language and which en/ch sample they pertain to
    - agri_ids contains the ids within processed_data.csv which are from the agriculture topic
- language_data.csv is a collected csv of all english and chichewa samples kept with their same rows and ids but en or ch cells shuffled into "query" or "answer" columns 

In [3]:
# Load the data with pandas and print head
translations = pd.read_csv('../Datasets/chichewa-machine-translation.csv')

In [4]:
print(translations.shape)
translations.head()

(14002, 5)


,sent_id,english,chichewa,topic,source
0,en1,Crops need to be planted early as recommended ...,Mbewu ziyenera kubzalidwa msanga potsatira mal...,agriculture,agriculture document
1,en2,"With the uncertainty in the rainfall pattern, ...","Ndi kusapanganika kwa kagwedwe ka mvula, mbewu...",agriculture,agriculture document
2,en3,Field preparation should be carried out prefer...,Minda isamalidwe pambuyo pomaliza kukolola pam...,agriculture,agriculture document
3,en4,This ensures deep ploughing and good decomposi...,Izi zimathandiza kuti khasu lidzilowa kwambiri...,agriculture,agriculture document
4,en5,"Where this is not practicable, farmers should ...","Ngati izi nzovuta kuchita, alimi ayetsetse kuk...",agriculture,agriculture document


In [5]:
# Helper to check format consistency for ids
# NOTE: checkIds works to format when all en and ch queries are in the same row (e.g. in the original dataset), but ruins question pairing in the ids after separation
def checkIds(df):
    # Assign sequential send_ids based on dataframe index (starting from 1).
    new_ids = []
    for idx, val in enumerate(df["sent_id"], start=1):
        prefix = "en"
        if isinstance(val, str) and len(val) >= 2:
            prefix = val[:2]
            # make sure prefix looks like letters, else fallback
            if not prefix.isalpha():
                prefix = "en"
        new_ids.append(f"{prefix}{idx}")
    df["sent_id"] = new_ids
    return df
translations = checkIds(translations)
translations

,sent_id,english,chichewa,topic,source
0,en1,Crops need to be planted early as recommended ...,Mbewu ziyenera kubzalidwa msanga potsatira mal...,agriculture,agriculture document
1,en2,"With the uncertainty in the rainfall pattern, ...","Ndi kusapanganika kwa kagwedwe ka mvula, mbewu...",agriculture,agriculture document
2,en3,Field preparation should be carried out prefer...,Minda isamalidwe pambuyo pomaliza kukolola pam...,agriculture,agriculture document
3,en4,This ensures deep ploughing and good decomposi...,Izi zimathandiza kuti khasu lidzilowa kwambiri...,agriculture,agriculture document
4,en5,"Where this is not practicable, farmers should ...","Ngati izi nzovuta kuchita, alimi ayetsetse kuk...",agriculture,agriculture document
...,...,...,...,...,...
13997,en13998,The disease spread too much.\n,Matendawo anafalikira\nkwambiri,Medical,Book of medical terms
13998,en13999,He lost a lot of blood.\n.,Anataya magazi ochuluka,Medical,Book of medical terms
13999,en14000,His body was too weak.\n,Thupi lake lina fooka kwambiri.,Medical,Book of medical terms
14000,en14001,His heart stopped beating.\n,Mtima wake unasiya kugunda.,Medical,Book of medical terms


In [6]:
# Check for duplicate ids, if empty then there are none
ids = translations["sent_id"]
print([q for q in translations[ids.isin(ids[ids.duplicated()])]["english"]])

[]


After checking for duplicates, the prep for language_data and processed_data begins to diverge. As such a new dataframe is defined here for language_data

In [7]:
language_df = translations.copy()
language_df = language_df.drop(["topic", "source"], axis=1)


Now that the language_df has been separated and before it has been randomly split for translations, an additional csv of manually transated phrases can be appended to the df

In [20]:
manualtranslations = pd.read_csv("../Datasets/ACADES_EngChichewa.csv")
manualtranslations

,Application\n(Web or Mobile),Page,English,ACADES Chichewa Translation
0,Desktop,Digital Asset Manager,Language,Chiyankhulo
1,Desktop,Digital Asset Manager,English,Chizungu
2,Desktop,Dashboard,Error,Pali Vuto!
3,Desktop,source-documents,Please be certain.,Chonde tsimikizani
4,Desktop,source-documents,Delete,Chotsani/Fufutani
...,...,...,...,...
127,Desktop,Settings,Background Image URL,Chithunzi
128,Desktop,Log in,Email address,Tsekulani ndi email
129,Mobile,UX,New Profile,Account yanu yanyuwani
130,Mobile,UX,Save Profile,Sungani account yanu


In [21]:
manualtranslations = manualtranslations.drop(["Application\n(Web or Mobile)", "Page", ], axis=1)
manualtranslations

,English,ACADES Chichewa Translation
0,Language,Chiyankhulo
1,English,Chizungu
2,Error,Pali Vuto!
3,Please be certain.,Chonde tsimikizani
4,Delete,Chotsani/Fufutani
...,...,...
127,Background Image URL,Chithunzi
128,Email address,Tsekulani ndi email
129,New Profile,Account yanu yanyuwani
130,Save Profile,Sungani account yanu


In [32]:
manualtranslations = manualtranslations.rename(columns={"English": "english", "ACADES Chichewa Translation": "chichewa"})
i = 14002
manualtranslations_ids = []
for idx, _ in enumerate(manualtranslations["english"], start=1):
    to_add = "en" + str(idx + i)
    manualtranslations_ids.append(to_add)
    i = i + 1
manualtranslations["sent_id"] = pd.Series(manualtranslations_ids)
manualtranslations

,english,chichewa,sent_id
0,Language,Chiyankhulo,en14003
1,English,Chizungu,en14005
2,Error,Pali Vuto!,en14007
3,Please be certain.,Chonde tsimikizani,en14009
4,Delete,Chotsani/Fufutani,en14011
...,...,...,...
127,Background Image URL,Chithunzi,en14257
128,Email address,Tsekulani ndi email,en14259
129,New Profile,Account yanu yanyuwani,en14261
130,Save Profile,Sungani account yanu,en14263


In [33]:
language_df = pd.concat([language_df, manualtranslations])
language_df

,sent_id,english,chichewa
0,en1,Crops need to be planted early as recommended ...,Mbewu ziyenera kubzalidwa msanga potsatira mal...
1,en2,"With the uncertainty in the rainfall pattern, ...","Ndi kusapanganika kwa kagwedwe ka mvula, mbewu..."
2,en3,Field preparation should be carried out prefer...,Minda isamalidwe pambuyo pomaliza kukolola pam...
3,en4,This ensures deep ploughing and good decomposi...,Izi zimathandiza kuti khasu lidzilowa kwambiri...
4,en5,"Where this is not practicable, farmers should ...","Ngati izi nzovuta kuchita, alimi ayetsetse kuk..."
...,...,...,...
127,en14257,Background Image URL,Chithunzi
128,en14259,Email address,Tsekulani ndi email
129,en14261,New Profile,Account yanu yanyuwani
130,en14263,Save Profile,Sungani account yanu


In [34]:
np.random.seed(610)
mask = np.random.rand(len(language_df)) < 0.5
language_df["query"] = np.where(mask, language_df["english"], language_df["chichewa"])
language_df["think"] = pd.Series()
language_df["answer"] = np.where(mask, language_df["chichewa"], language_df["english"])
language_df["response"] = pd.Series()

qa_df = language_df[["sent_id", "query", "think", "answer", "response"]]

qa_ratio = (qa_df["query"] == language_df["english"]).mean()
print(qa_ratio)

qa_df

0.50148577897269


,sent_id,query,think,answer,response
0,en1,Crops need to be planted early as recommended ...,NaN,Mbewu ziyenera kubzalidwa msanga potsatira mal...,NaN
1,en2,"With the uncertainty in the rainfall pattern, ...",NaN,"Ndi kusapanganika kwa kagwedwe ka mvula, mbewu...",NaN
2,en3,Field preparation should be carried out prefer...,NaN,Minda isamalidwe pambuyo pomaliza kukolola pam...,NaN
3,en4,Izi zimathandiza kuti khasu lidzilowa kwambiri...,NaN,This ensures deep ploughing and good decomposi...,NaN
4,en5,"Where this is not practicable, farmers should ...",NaN,"Ngati izi nzovuta kuchita, alimi ayetsetse kuk...",NaN
...,...,...,...,...,...
127,en14257,Chithunzi,NaN,Background Image URL,NaN
128,en14259,Tsekulani ndi email,NaN,Email address,NaN
129,en14261,Account yanu yanyuwani,NaN,New Profile,NaN
130,en14263,Sungani account yanu,NaN,Save Profile,NaN


In [35]:
qa_df.to_csv("language_data.csv", index=False)

Each row has a unique sent_id with a number and "en", so it doesn't give much more information than any generic id. However, because the en and ch versions of each query will be split up, these ids will be used in the experience format either as-is for the english ones or after replacing "en" with "ch" for their chichewa counterparts. Given that the ids are kept the same and all queries across topics will be used for training, topic and source columns can be removed.

NOTE: Because of the range of topics, oversampling will be done on the agriculture ones. Other options include running a second pass of fine-tuning just on the agricultural data, and to add topic metadata.

In [ ]:
# Before removing topic, oversample the agriculture topic rows to accentuate the agricultural importance of the training data
# This will be done by making a separate agriculture dataframe, saving the ids of the agriculture queries, and recombining all of the dataframes.
df = translations.loc[translations["topic"] != "agriculture"]
agriculture_df = translations.loc[translations["topic"] == "agriculture"]

In [ ]:
en_df = df[["sent_id", "english"]]
print(en_df)
ch_df = df[["sent_id", "chichewa"]]
print(ch_df)

In [ ]:
agriculture_en_df = agriculture_df[["sent_id", "english"]]
print(agriculture_en_df)
agriculture_ch_df = agriculture_df[["sent_id", "chichewa"]]
print(agriculture_ch_df)


Before more processing is done with these for the general experience data, these will be shuffled and recombined in rows for the 

In [ ]:
ch_df["sent_id"] = ch_df["sent_id"].str.replace("en", "ch")
ch_df

In [ ]:
agriculture_ch_df["sent_id"] = agriculture_ch_df["sent_id"].str.replace("en", "ch")
agriculture_ch_df

In [ ]:
df = pd.concat([en_df, ch_df], axis=0)
df["query"] = df["english"].fillna(df["chichewa"])
df = df.drop(["english", "chichewa"], axis=1)
df # The dataframe with all non-agricultural queries

In [ ]:
agriculture_df = pd.concat([agriculture_en_df, agriculture_ch_df], axis=0)
agriculture_df["query"] = agriculture_df["english"].fillna(agriculture_df["chichewa"])
agriculture_df = agriculture_df.drop(["english", "chichewa"], axis=1)
agriculture_df # dataframe with all agricultural queries

In [ ]:
agri_ids = agriculture_df["sent_id"].copy()
old_agri_ids = agri_ids.copy()
duplicates_num = len([q for q in agriculture_df[agri_ids.isin(agri_ids[agri_ids.duplicated()])]["query"]])
agriculture_dupes_num = agriculture_df.shape[0]
print("There are ", duplicates_num, " duplicates in the set of ", agriculture_dupes_num, "agriculture queries")

When separated into english and chichewa the total number of query samples is 28,004 (14,002 english and 14,002 chichewa) and 21.4% (5986) agriculture samples.

For best results practicing agricultural information, the agricultural prompts will be oversampled, which requires duplicates of each query and new corresponding ids (starting at 14001, the highest number from the original dataset). It's important that the numbers remain the same for corresponding en/ch pairs. 

NOTE: It may prove more effective when grading and/or fine-tuning to keep the english and chichewa question-answer pairs together. As long as the number part of the id remains the same, then that can be done at a later point even after the initial answers have been produced and graded. 

In [ ]:
# Create a new series of ids which will fit with the oversampled duplicates of the original agriculture queries. 14002 is because it will be tacked onto the end of the original 14002 rows.
new_ids = pd.Series([str(id[:2] + str(int(id[2:]) + 14002)) for id in agri_ids])
print(new_ids)
dupli_df = agriculture_df.copy(deep=True)
dupli_df["sent_id"] = dupli_df["sent_id"].apply(lambda id: "".join([id[:2], str(int(id[2:]) + 14002)]))
print(dupli_df)

In [ ]:
agri_ids = pd.concat([agri_ids, new_ids])
print(old_agri_ids.shape)
print(agri_ids)
agri_ids.to_csv("agri_ids.csv", index=False)
dupli_df # duplicate of agriculture_df but with updated ids

In [ ]:
agriculture_df

After oversampling, join the agro and original dataframes and apply prompt templates to each query to make them fit into the question-answer format.

In [ ]:
processed_df = pd.concat([agriculture_df, df, dupli_df], axis=0) #combine all three dataframes
print(agriculture_df.shape)
print(df.shape)
print(dupli_df.shape)
processed_df

In [ ]:
print(processed_df.shape)
len(set(processed_df["sent_id"]))
processed_df[:20]

Now that the dataframes are joineed we have 33990 rows of queries, 50:50 split between english and chichewa, with 35.2% agricultural specific queries. 

The next step to be able to pass them into answer generation and save to a csv for scoring. To do this, every query has to be actually phrased as a query, so templates will be used to rephrase the facts in the dataset based on the en or ch id. The template additions are useful to contextualize the informtion, but the usefulness of some of the queries with these templates is still suspect. If retraining has to happen, expansion of templates might be wise.

In [ ]:
templates = {
    "en": [
        "What should I understand from this: '{}'",
        "Can you explain whether this statement is correct and why: '{}'",
        "How would you respond if someone told you: '{}'",
        "A farmer says: '{}'. How would you advise them?"
    ],
    "ch": [
        "Kodi ndimvetsetse chiyani pa izi: '{}'",
        "Kodi mungandifotokozere ngati mawu awa ndi olondola komanso chifukwa chake: '{}'",
        "Kodi mungayankhe bwanji ngati munthu wina atakuwuzani kuti: '{}'",
        "Mlimi akuti: '{}'. Kodi mungamupatse upangiri wotani?"
    ]
}

def apply_template(row):
    lang = row['sent_id'][:2]  # 'en' or 'ch'
    template = random.choice(templates[lang])
    return template.format(row['query'])


In [ ]:
processed_df["query"] = processed_df.apply(apply_template, axis=1)

In [ ]:
ids = processed_df["sent_id"]
duplicates_num = len([q for q in processed_df[ids.isin(ids[ids.duplicated()])]["query"]])
agriculture_dupes_num = agriculture_df.shape[0] * 2
print(duplicates_num)
print(agriculture_dupes_num)
if duplicates_num == agriculture_dupes_num:
    print("There are the correct number of duplicates after oversampling")
elif duplicates_num > agriculture_dupes_num:
    print("There are more duplicates than expected")
elif duplicates_num < agriculture_dupes_num:
    print("There are fewer duplicates than expected")

In [ ]:
processed_df.to_csv("processed_data.csv", index=False)

The data is now stored in our desired format to a csv file such that it can be processed by the '/batch_test' endpoint in testing.py

For cost and time estimates, it's important to try sampling some of the queries and tokenizing them to create an average token size. This average can be used to estimate the required number of tokens as well as the number of requests which must be made to generate the query answers. 

In [ ]:
def myTokenizer(text):
    return word_tokenize(text)
def querySampler(i, j):
    querySample = [q for q in processed_df["query"][i:j]]
    diff = int(j - i)
    c_length = sum(len(q) for q in querySample)/diff
    print(f"Estimated character length of queries from samples {i} to {j}: ", c_length)
    vectorizer = CountVectorizer(tokenizer=myTokenizer)
    X = vectorizer.fit_transform(querySample)
    token_num = len(vectorizer.get_feature_names_out())/diff
    print(f"Average number of tokens per sample from samples {i} to {j}: ", token_num)
    return c_length, token_num

In [ ]:
reasoningprompt = """ 
A conversation between User and Assistant. The user asks a question, and the assistant solves it.
The assistant first thinks about the reasoning process in the mind and then provides the user
with the answer. The reasoning process and answer are enclosed within <think></think> and <answer></answer> tags
respectively, i.e., <think> reasoning process here </think> and <answer> answer here </answer>.

Do not generate new code. Do not write python code.

You may also be given examples by the user telling you the expected response format.
Follow the format of the examples, but solve the specific problem asked by the user, not the examples.

Very important - Remember again, your output format should be:
<think> reasoning process here </think>
<answer> answer here </answer>

Your response will be scored by extracting the substring between the <answer>...</answer> tags.
It is critical to follow the above format.
feature_extraction_utilsling to follow the response format will result in a penalty."""
vectorizer = CountVectorizer(tokenizer=myTokenizer)
X = vectorizer.fit_transform([reasoningprompt])
print("Reasoning prompt token #: ", len(vectorizer.get_feature_names_out()))
vectorizer.get_feature_names_out()

In [ ]:
a, b = querySampler(0, 20)
c, d = querySampler(55, 75)
e, f = querySampler(966, 1000)
print("Total average sampled number of characters: ", (a + c + e)/3)
print("Total average number of sampled tokens: ", (b + d + f)/3)

From these few samples, the token size for the whole dataset is estimated to be 501,692 from 14.76 sampled average spread across 33,990 queries. This only takes the inputs into account per token count, discounts the reasoning prompt, and means that the final number would be n(501,962) where n is the number of answers generated per query. For now assuming n is 3 then the total token count would be ~1,505,077 spread across 101,970 requests. 

Passing through GPT-4.1 mini then the price per million tokens under batch api conditions at time of writing is 20 cents while the output is 80 cents. Using n=3 allows a single request per query and reduces the number of tokens back down to 1/3rd of the previous estimate.

Factoring in the token size of the reasoning prompt (~203 tokens) and assuming it is added in the same way before every query, the average goes up to ~253 tokens which makes a total of ~8,599,470 tokens for one batch spread across the whole dataset.

In [ ]:
queries = processed_df
for i, q in zip(queries["sent_id"][:20], queries["query"][:20]):
    print(i)
    print(q)

## ACADES + Cohesion X AIFarmer data sampling

Description: The existing run of AIFarmer using Cohesion X provides access to threads of messages and individual messages of the same sort as the DigitalGreen Agriqueries. In this case the gathering of the data requires additional anonymizing and cleaning to identify which messages are questions/responses etc. This is useful as it's a direct example of questions that are being asked in the field, and there is a mixture of Chichewa and English questions. 

Data cleaning here mainly requires:
- Anonymizing data
- 

In [ ]:
# Load the data with pandas and print head
